# 🩻 Hands-on: Agentes de IA para Radiologistas

### Jornada Paulista de Radiologia 2026

**Eduardo Moreno Judeice de Mattos Farina** · UNIFESP · Hospital Israelita Albert Einstein

---

> Sessão prática de ~75 minutos para radiologistas curiosos sobre como construir agentes de IA. Zero Python exigido.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eduardofarina/hands-on-agentes-jpr2026/blob/main/hands_on_agentes_jpr2026.ipynb)

## 📋 Índice

0. [Boas-vindas e Setup](#secao-0) — como rodar o notebook, como configurar sua API key
1. [O que é um agente de IA?](#secao-1) — chatbot vs. agente, o loop, os 4 componentes
2. [Prompt Engineering: o Residente Virtual](#secao-2) — system prompt, persona, formato estruturado
3. [Context Engineering e RAG: Consultando Diretrizes](#secao-3) — por que o LLM precisa "ler" ao invés de "saber"
4. [Tools Externas: PubMed ao Vivo](#secao-4) *(bônus)* — literatura sob demanda
5. [Agente Multimodal: Lendo uma Radiografia](#secao-5) ⭐ — visão + saída estruturada
6. [Quebrando o Agente: Limites e Guardrails](#secao-6) — alucinação, escopo, responsabilidade
7. [Encerramento](#secao-7) — recap, quando **não** usar agente, próximos passos

**Apêndices (colapsados):** Troubleshooting · Como obter API key · Glossário · Referências

## 🎯 Objetivos

Ao final desta sessão, você será capaz de:

1. **Explicar** a diferença entre um chatbot e um agente de IA.
2. **Identificar** os 4 componentes essenciais de um agente: modelo, instruções, ferramentas e memória/contexto.
3. **Distinguir** *prompt engineering* (o que você escreve) de *context engineering* (o que entra na janela de contexto).
4. **Reconhecer** quando vale a pena usar um agente — e quando um prompt simples basta.
5. **Modificar** um agente pronto (trocar instruções, adicionar uma ferramenta) sem quebrar nada.
6. **Enxergar os limites**: alucinação, custo, latência, e por que *nenhum* destes agentes substitui um laudo real.

<a id='secao-0'></a>
# 0 · Boas-vindas e Setup

### ⏱️ ~5 minutos

Este notebook é um **documento interativo**. Ele mistura texto (como este que você está lendo) com pequenos blocos de código Python que você pode *executar* clicando no botão ▶️ que aparece ao lado esquerdo de cada célula de código.

**Você não precisa saber Python.** Os exemplos já estão prontos. Em vários pontos do notebook você vai encontrar uma caixa **"🔧 Experimente"** — aí sim te convido a mudar uma linha ou outra e rodar de novo pra ver o que acontece.

**Como rodar uma célula de código:**
1. Clique na célula (ela fica destacada).
2. Aperte **Shift + Enter** *ou* clique no botão ▶️ à esquerda.
3. Espere aparecer um ✅ verde ou o resultado logo abaixo.

Se der erro, não se assuste — a seção [Troubleshooting](#apendice-a) no final tem as soluções mais comuns.

### 🔑 Configurando sua API key

Um agente precisa de um **modelo de linguagem** por trás — neste notebook vamos usar o **Google Gemini**, que tem um tier gratuito generoso e suporta imagens (importante pra seção da radiografia).

**Você já deveria ter feito isso antes da sessão.** Se não, siga estes passos agora:

1. Abra <https://aistudio.google.com/apikey> em outra aba (login com sua conta Google normal).
2. Clique em **"Create API key"** → copie a chave gerada (começa com `AIza...`).
3. **Aqui no Colab**, olhe a barra lateral esquerda e clique no ícone de **🔑 chave** (*Secrets*).
4. Clique em **"+ Add new secret"** e preencha:
   - **Name**: `GOOGLE_API_KEY` (exatamente assim, com letras maiúsculas)
   - **Value**: cole sua chave
   - Ative o toggle **"Notebook access"**
5. Pronto. Rode a próxima célula.

> ⚠️ **Por que não colar a chave direto no código?** Se você compartilhar o notebook (e vai querer compartilhar), a chave vai junto e qualquer pessoa pode usar sua cota. O sistema de *Secrets* mantém a chave fora do código.

In [ ]:
# @title ▶️ Rode esta célula primeiro (instala tudo e configura a API key) { display-mode: "form" }

print('⏳ Instalando dependências... (vai levar 1-2 minutos na primeira vez)')

import subprocess, sys
_pkgs = ['agno', 'google-genai', 'lancedb', 'tantivy', 'pypdf', 'reportlab', 'requests', 'pillow']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *_pkgs], check=True)

import os

# Carrega a API key do Secrets do Colab (ou de variável de ambiente local)
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
except Exception:
    if not os.environ.get('GOOGLE_API_KEY'):
        raise RuntimeError(
            '❌ GOOGLE_API_KEY não encontrada.\n'
            'No Colab: ícone 🔑 (Secrets) na barra esquerda → + Add new secret →\n'
            'Name = GOOGLE_API_KEY, Value = sua chave, ative "Notebook access".'
        )

# Evita conflito quando o Colab também tem GEMINI_API_KEY definido
# (o google-genai emite warning e isso atrapalha o stream do Agno).
os.environ.pop('GEMINI_API_KEY', None)

# Teste rápido: uma chamada ao Gemini pra validar a chave antes de seguir
from google import genai
_client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
_test = _client.models.generate_content(
    model='gemini-2.5-flash',
    contents='Responda apenas com a palavra: PRONTO',
)
assert 'PRONTO' in _test.text.upper(), f'Resposta inesperada: {_test.text!r}'

# ── Helper Colab-friendly para rodar agentes ──────────────────────────────
# O `print_response` do Agno usa Rich Live, que não redesenha bem no Colab
# (a célula trava no spinner mesmo com o agente já tendo terminado).
# Esta função usa `agent.run()` + display(Markdown) — render estável e
# qualquer erro fica visível (não fica engolido pelo Rich).
def show_response(agent, prompt, **kwargs):
    """Roda um agente e mostra tool calls + resposta no Colab."""
    import time, traceback
    from IPython.display import display, Markdown

    print('⏳ Rodando agente...')
    t0 = time.time()
    try:
        response = agent.run(prompt, **kwargs)
    except Exception as e:
        print(f'❌ Erro depois de {time.time()-t0:.1f}s: {type(e).__name__}: {e}')
        traceback.print_exc()
        return None
    print(f'✅ Pronto em {time.time()-t0:.1f}s')

    # Tool calls (a API muda entre versões do Agno; tentamos vários atributos)
    tools = getattr(response, 'tools', None) or []
    for tc in tools:
        name = getattr(tc, 'tool_name', None) or getattr(tc, 'name', '?')
        args = getattr(tc, 'tool_args', None) or getattr(tc, 'arguments', None) or {}
        result = getattr(tc, 'result', None) or getattr(tc, 'tool_call_result', None) or ''
        result_str = str(result)
        if len(result_str) > 400:
            result_str = result_str[:400] + ' ...'
        print(f'🔧 {name}({args})')
        print(f'   → {result_str}')

    content = getattr(response, 'content', None) or '_(sem conteúdo)_'
    display(Markdown(content))
    return response

# Criar pastas que vamos usar
os.makedirs('data', exist_ok=True)
os.makedirs('assets', exist_ok=True)
os.makedirs('tmp/lancedb', exist_ok=True)

print('✅ Tudo pronto! Pode ir para a seção 1.')

<a id='secao-1'></a>
# 1 · O que é um agente de IA?

### ⏱️ ~10 minutos

Provavelmente você já usou o ChatGPT ou o Gemini direto no navegador. Isso é um **chatbot**: você manda uma pergunta, o modelo responde, fim.

Um **agente** é diferente. Um agente é um modelo de linguagem que:

1. **Recebe um objetivo** (não só uma pergunta isolada).
2. **Tem acesso a ferramentas** (*tools*) — funções que ele pode *decidir* chamar: uma calculadora, uma busca, uma API, um banco de dados.
3. **Roda em um loop**: pensa → chama uma tool → lê o resultado → pensa de novo → chama outra tool → ... → entrega a resposta final.

Em um diagrama bem simples:

```
          ┌─────────────┐
  você ──►│   Agente    │──► resposta
          │  (modelo)   │
          └──────┬──────┘
                 │ decide chamar uma tool
                 ▼
          ┌─────────────┐
          │    Tool     │  ← calculadora, API, banco, etc.
          └──────┬──────┘
                 │ devolve o resultado
                 ▼
          ┌─────────────┐
          │   Agente    │  ← lê, pensa, decide o próximo passo
          └─────────────┘
```

### Os 4 componentes essenciais de um agente

| Componente | O que é | Analogia clínica |
|---|---|---|
| **Modelo** | O LLM que raciocina (Gemini, GPT, Claude...) | O cérebro do residente |
| **Instruções** | O *system prompt* que define persona, regras, formato | O briefing de plantão |
| **Tools** | Funções que o agente pode chamar | Consultar PACS, tabela de protocolos, ligar pro staff |
| **Memória/Contexto** | O que o agente "lembra" da conversa e o que ele tem acesso | O prontuário + o que você falou na última frase |

Vamos ver isso na prática com um exemplo que radiologista enfrenta todo dia — **escolher o protocolo certo de um exame** — onde o LLM sozinho pode chutar, e por que dar uma tool resolve.

### 1a · Chat puro: escolha de protocolo (e por que o LLM chuta)

**Cenário clínico real do dia-a-dia:** o médico assistente pede *"TC de abdome para estadiamento"*. Você, na sala de protocolo, precisa decidir **quais fases adquirir**:

- **Sem contraste?** (ex: busca de hemorragia, cálculos, avaliação de esteatose)
- **Fase arterial?** (ex: lesões hipervasculares — HCC, metástases de melanoma/NET/CCR/tireoide)
- **Fase portal?** (avaliação parenquimatosa geral — quase sempre presente)
- **Fase tardia/excretora?** (ex: via urinária, colangiocarcinoma, caracterização de lesão)

Essa escolha **importa muito**: fases a mais = dose desnecessária e tempo de sala; fases a menos = estadiamento incompleto e repetição de exame. O **ACR Appropriateness Criteria** e os protocolos institucionais orientam essas decisões por indicação clínica.

**Pergunta para o LLM:**

> *"Paciente com melanoma metastático. Pedido: TC de abdome para estadiamento. Quais fases devo adquirir?"*

Vamos perguntar **direto pro modelo**, sem tool nenhuma. Observe: a resposta é consistente? Ele cita fonte? Se rodarmos de novo, dá a mesma resposta?

In [ ]:
# Chamada direta ao Gemini, SEM agente e SEM tool.
# Só o modelo bruto respondendo.

from google import genai

client = genai.Client()   # usa GOOGLE_API_KEY do ambiente

question = (
    'Paciente com melanoma metastático. O assistente pediu TC de abdome '
    'para estadiamento. Quais fases de aquisição devo obter (sem contraste, '
    'arterial, portal, tardia)? Responda objetivamente em até 5 linhas.'
)

print('──── Rodada 1 ────')
r1 = client.models.generate_content(model='gemini-2.5-flash', contents=question)
print(r1.text)

print('\n──── Rodada 2 (mesma pergunta) ────')
r2 = client.models.generate_content(model='gemini-2.5-flash', contents=question)
print(r2.text)

print(
    '\n⚠️  Repare: resposta plausível, mas '
    'SEM fonte, SEM rastreabilidade, e possivelmente DIFERENTE entre as duas rodadas.'
)

### 1b · Mesmo problema, agora com um agente + tool

Agora vamos criar um **agente** de verdade, com uma **tool** `get_ct_protocol(indication, body_region)` que consulta uma **tabela estruturada** de protocolos (baseada em ACR Appropriateness Criteria + protocolo institucional simplificado). O agente **decide sozinho** chamar a tool, lê o protocolo, e devolve a resposta com fases + justificativa + referência.

> 📝 **Nota pedagógica:** a tabela que vamos usar é uma versão **simplificada** para fins didáticos. Protocolos reais são definidos localmente por cada serviço e revisados periodicamente. O ponto aqui é mostrar como **conhecimento estruturado** (uma tabela, uma regra, uma API) se transforma em uma *tool* que o agente consulta — e como isso torna a resposta **auditável** (você pode abrir a tabela e ver de onde veio).

In [ ]:
# ── Tabela de protocolos (simplificada, educacional) ──────────────────────
# Baseada em ACR Appropriateness Criteria + padrões institucionais comuns.
# Cada entrada: indicação → {phases, rationale, acr_topic}

PROTOCOLS = {
    ('melanoma staging', 'abdome'): {
        'phases': ['sem contraste', 'arterial', 'portal'],
        'rationale': 'Metástases de melanoma são frequentemente hipervasculares; '
                     'a fase arterial aumenta a sensibilidade para pequenas lesões '
                     'hepáticas, e a fase sem contraste ajuda a diferenciar '
                     'hemorragia/calcificação.',
        'acr_topic': 'ACR AC — Staging of Known Malignancies (Melanoma)',
    },
    ('colorretal staging', 'abdome'): {
        'phases': ['portal'],
        'rationale': 'Metástases colorretais são tipicamente hipovasculares e melhor '
                     'caracterizadas na fase portal. Fases adicionais acrescentam '
                     'dose sem ganho diagnóstico.',
        'acr_topic': 'ACR AC — Colorectal Cancer Staging',
    },
    ('hcc staging', 'abdome'): {
        'phases': ['sem contraste', 'arterial', 'portal', 'tardia (3 min)'],
        'rationale': 'LI-RADS exige protocolo multifásico para caracterização de '
                     'nódulos em paciente cirrótico (wash-in arterial e wash-out '
                     'portal/tardio).',
        'acr_topic': 'ACR LI-RADS v2018',
    },
    ('adenocarcinoma pancreatico', 'abdome'): {
        'phases': ['fase pancreática (40-50s)', 'portal'],
        'rationale': 'A fase pancreática parenquimatosa maximiza o contraste entre '
                     'o tumor hipoatenuante e o parênquima pancreático normal; '
                     'a portal avalia fígado e mesentério.',
        'acr_topic': 'ACR AC — Pancreatic Adenocarcinoma',
    },
    ('caracterizacao de massa renal', 'abdome'): {
        'phases': ['sem contraste', 'corticomedular', 'nefrográfica', 'excretora'],
        'rationale': 'Protocolo renal de 4 fases permite calcular realce absoluto '
                     '(≥20 UH sugere lesão sólida) e avaliar via coletora.',
        'acr_topic': 'ACR AC — Indeterminate Renal Mass',
    },
    ('trauma contuso', 'abdome'): {
        'phases': ['portal'],
        'rationale': 'Fase portal única é padrão para trauma estável. Adicionar '
                     'fase arterial se houver suspeita de sangramento ativo ou '
                     'instabilidade hemodinâmica (protocolo dual).',
        'acr_topic': 'ACR AC — Major Blunt Trauma',
    },
    ('abdome agudo', 'abdome'): {
        'phases': ['portal'],
        'rationale': 'Fase portal única cobre a maioria das causas (apendicite, '
                     'diverticulite, obstrução, isquemia). Casos selecionados '
                     '(isquemia mesentérica) podem exigir arterial.',
        'acr_topic': 'ACR AC — Right Lower Quadrant Pain',
    },
    ('nodulo adrenal incidental', 'abdome'): {
        'phases': ['sem contraste', 'portal', 'tardia (15 min)'],
        'rationale': 'Protocolo de adrenal com cálculo de wash-out absoluto e '
                     'relativo permite caracterizar adenoma vs. lesão não-adenoma.',
        'acr_topic': 'ACR Incidental Findings — Adrenal',
    },
}


def _normalize(s: str) -> str:
    import unicodedata
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    return s.lower().strip()


def get_ct_protocol(indication: str, body_region: str) -> dict:
    """Retorna o protocolo de TC recomendado para uma indicação clínica.

    Consulta uma tabela institucional simplificada (baseada em ACR Appropriateness
    Criteria) e devolve as fases a adquirir, a justificativa clínica e a referência.

    Args:
        indication (str): Indicação clínica em linguagem livre, ex: 'melanoma staging',
            'estadiamento de câncer colorretal', 'caracterização de massa renal'.
        body_region (str): Região anatômica do exame, ex: 'abdome', 'tórax'.

    Returns:
        dict: {
            'phases': list[str] — fases a adquirir,
            'rationale': str — justificativa clínica,
            'acr_topic': str — tópico ACR de referência,
            'matched_indication': str — indicação canônica encontrada na tabela,
        }
        Se a indicação não for reconhecida, retorna fallback pedindo revisão
        do radiologista.
    """
    ind_norm = _normalize(indication)
    reg_norm = _normalize(body_region)

    # Mapa de sinônimos simples para aumentar a chance de match
    aliases = {
        'melanoma': 'melanoma staging',
        'estadiamento de melanoma': 'melanoma staging',
        'colorretal': 'colorretal staging',
        'estadiamento de colorretal': 'colorretal staging',
        'cancer colorretal': 'colorretal staging',
        'cancer de colon': 'colorretal staging',
        'hcc': 'hcc staging',
        'hepatocarcinoma': 'hcc staging',
        'cirrotico': 'hcc staging',
        'pancreas': 'adenocarcinoma pancreatico',
        'cancer de pancreas': 'adenocarcinoma pancreatico',
        'massa renal': 'caracterizacao de massa renal',
        'lesao renal indeterminada': 'caracterizacao de massa renal',
        'trauma': 'trauma contuso',
        'trauma abdominal': 'trauma contuso',
        'abdome agudo': 'abdome agudo',
        'apendicite': 'abdome agudo',
        'adrenal': 'nodulo adrenal incidental',
        'incidentaloma adrenal': 'nodulo adrenal incidental',
    }

    # Tenta match direto, depois por alias (substring)
    canonical = None
    for key in PROTOCOLS:
        if key[0] in ind_norm:
            canonical = key[0]
            break
    if canonical is None:
        for alias, target in aliases.items():
            if alias in ind_norm:
                canonical = target
                break

    if canonical and (canonical, reg_norm) in PROTOCOLS:
        entry = PROTOCOLS[(canonical, reg_norm)]
        return {**entry, 'matched_indication': canonical}

    return {
        'phases': None,
        'rationale': 'Indicação não encontrada na tabela institucional simplificada. '
                     'Sugerir revisão pelo radiologista responsável antes de definir '
                     'o protocolo.',
        'acr_topic': 'N/A',
        'matched_indication': None,
    }


# ── Agente que usa a tool ────────────────────────────────────────────────
from agno.agent import Agent
from agno.models.google import Gemini

protocol_agent = Agent(
    model=Gemini(id='gemini-2.5-flash'),
    tools=[get_ct_protocol],
    instructions=(
        'Você é um assistente de sala de protocolo radiológico. '
        'Dada uma indicação clínica, SEMPRE chame a tool get_ct_protocol '
        'para descobrir o protocolo, e explique o resultado ao técnico em linguagem '
        'clara e objetiva. Inclua as fases recomendadas, a justificativa clínica '
        'e a referência (tópico ACR) retornadas pela tool. '
        'Se a tool retornar matched_indication=None, NÃO invente um protocolo — '
        'diga que não há protocolo definido na tabela e sugira consultar o radiologista.'
    ),
    markdown=True,
)

show_response(protocol_agent, 'Paciente com melanoma metastático. Pedido: TC de abdome para estadiamento. '
    'Quais fases devo adquirir?'
)

> 💡 **O que aconteceu:** o agente leu a pergunta, identificou a indicação clínica (*melanoma staging*), chamou a tool `get_ct_protocol`, pegou o protocolo estruturado da tabela, e devolveu uma resposta **consistente, rastreável e com fonte**. É esse **loop** — pensar, agir, ler, pensar de novo — que define um agente.

**Compare com a resposta 1a:** o LLM puro dá uma resposta plausível, mas sem garantias. O agente 1b te dá algo que você pode **auditar**: a tabela existe, o protocolo veio de lá, e se o padrão institucional mudar, você edita a tabela — não precisa re-treinar modelo nenhum.

### 🔧 Experimente

Mude a indicação clínica e rode de novo. Observe como o agente escolhe fases diferentes para indicações diferentes. Algumas ideias:

- **"estadiamento de câncer colorretal"** → deveria pedir só fase portal
- **"caracterização de massa renal"** → protocolo renal com 4 fases
- **"suspeita de HCC em paciente cirrótico"** → protocolo multifásico hepático (LI-RADS)
- **"trauma contuso de abdome"** → fase portal (± arterial)
- **Uma indicação fora da tabela** (ex: "dor torácica atípica") → veja o agente usar o fallback

In [ ]:
# 🔧 MUDE AQUI ↓
clinical_indication = 'estadiamento de melanoma metastático'   # tente outras indicações
body_region = 'abdome'
# 🔧 NÃO precisa mexer no resto ↓↓↓

show_response(protocol_agent, f'Indicação clínica: {clinical_indication}. '
    f'Região: {body_region}. Qual o protocolo recomendado?'
)

<a id='secao-2'></a>
# 2 · Prompt Engineering: o Residente Virtual

### ⏱️ ~15 minutos

**Prompt engineering** é a arte de escrever instruções que fazem o modelo se comportar do jeito que você quer. Em um agente, a parte mais importante é o **system prompt** (as *instructions*) — o texto que você dá no momento de criar o agente, e que ele carrega em toda interação.

Um bom system prompt define:

- **Persona**: quem o agente "é" (ex: "você é um residente R3 de radiologia torácica").
- **Escopo**: o que ele pode e o que ele NÃO pode fazer.
- **Formato de saída**: como a resposta deve ser estruturada.
- **Restrições**: tom, idioma, limites, fontes que pode citar.
- **Comportamento em caso de incerteza**: "se não tiver certeza, diga 'não sei' ao invés de inventar".

Vamos construir um **"residente virtual de tórax"** que recebe achados soltos e devolve um laudo estruturado.

### 2 · Residente Virtual de Tórax

**Input (exemplo):**

> "Paciente masculino, 67 anos, tabagista 40 maços-ano. TC de tórax: nódulo sólido espiculado em lobo superior direito, 22 mm, sem calcificação, sem gordura. Demais campos pulmonares sem alterações. Mediastino sem linfonodomegalias. Ausência de derrame pleural."

**Output esperado:** laudo estruturado em seções (Técnica / Achados / Impressão) + **Lung-RADS sugerido** + recomendação de seguimento.

In [ ]:
from agno.agent import Agent
from agno.models.google import Gemini

CHEST_RESIDENT_PROMPT = """\
Você é um residente R3 de radiologia torácica, brasileiro, treinado em
protocolos da SBR/Colégio Brasileiro de Radiologia e Fleischner Society.

TAREFA: receber achados radiológicos descritos em linguagem livre e gerar um
laudo estruturado educacional.

FORMATO DE SAÍDA (sempre nesta ordem, use estes títulos exatos):
  ## Técnica
  ## Achados
  ## Impressão
  ## Lung-RADS sugerido
  ## Recomendação de seguimento

REGRAS:
- Escreva em português claro, técnico mas acessível.
- Na seção 'Lung-RADS sugerido', indique a categoria (1-4X) com breve justificativa.
- Na seção 'Recomendação de seguimento', cite o intervalo baseado no Fleischner 2017
  ou Lung-RADS.
- Se os achados forem insuficientes, diga explicitamente o que falta.
- SEMPRE termine com: '⚠️ Material educacional — não substitui laudo por radiologista.'
"""

chest_agent = Agent(
    model=Gemini(id='gemini-2.5-flash'),
    instructions=CHEST_RESIDENT_PROMPT,
    markdown=True,
)

findings = (
    'Paciente masculino, 67 anos, tabagista 40 maços-ano. '
    'TC de tórax: nódulo sólido espiculado em lobo superior direito, 22 mm, '
    'sem calcificação, sem gordura. Demais campos pulmonares sem alterações. '
    'Mediastino sem linfonodomegalias. Ausência de derrame pleural.'
)

show_response(chest_agent, findings)

### 🔧 Experimente

Duas coisas pra testar:

1. **Troque a persona**: mude `agent_persona` para "residente de neurorradiologia" e passe um achado de neuro (ex: área hipodensa em território da ACM direita). Veja como o estilo, as siglas e os scores mudam (ASPECTS no lugar de Lung-RADS).
2. **Troque o formato**: mude `output_format` para JSON e veja como a saída muda — isso é o que você faria se fosse integrar com outro sistema.

In [ ]:
# 🔧 MUDE AQUI ↓
agent_persona = 'residente R3 de radiologia torácica'   # tente: 'residente de neurorradiologia'
clinical_findings = '''
Paciente masculino, 67 anos, tabagista 40 maços-ano.
TC de tórax: nódulo sólido espiculado em lobo superior direito, 22 mm.
'''
output_format = 'markdown com seções (Técnica, Achados, Impressão, Score, Seguimento)'
# tente: 'JSON estrito com as chaves technique, findings, impression, score, followup'
# 🔧 NÃO precisa mexer no resto ↓↓↓

custom_agent = Agent(
    model=Gemini(id='gemini-2.5-flash'),
    instructions=(
        f'Você é um {agent_persona} brasileiro. '
        f'Receba achados e devolva um laudo estruturado no formato: {output_format}. '
        'Inclua o score/escala apropriada à sub-especialidade. '
        'Termine com: "⚠️ Material educacional — não substitui laudo."'
    ),
    markdown=True,
)
show_response(custom_agent, clinical_findings)

<a id='secao-3'></a>
# 3 · Context Engineering e RAG: Consultando Diretrizes

### ⏱️ ~15 minutos

### Prompt engineering ≠ Context engineering

- **Prompt engineering** = o que *você escreve* para o modelo (instruções, exemplos).
- **Context engineering** = o que *entra na janela de contexto* do modelo em cada chamada — e isso inclui instruções, histórico, documentos recuperados, resultados de tools, memória de longo prazo...

A janela de contexto é **finita** (dezenas a centenas de milhares de tokens) e **cara** (quanto mais tokens, mais demora e mais custa). Gerenciar *o que entra nela* é uma das decisões mais importantes ao construir um agente.

### Por que RAG?

Um LLM **não sabe** — ele foi treinado até uma certa data e depois disso não aprendeu mais nada. Pior: mesmo o que ele "sabe" pode estar desatualizado ou errado. Se você pergunta sobre uma diretriz específica da ACR, ele pode:

- **Chutar** (alucinação) — dá uma resposta convincente mas errada.
- **Dizer que não sabe** (se bem instruído).
- **Ler a diretriz de verdade e citar** — se você der a ele.

A terceira opção é **RAG** (*Retrieval Augmented Generation*): você indexa os documentos que importam, e antes de cada pergunta o sistema busca os trechos relevantes e coloca na janela de contexto do modelo. O LLM vira um **leitor**, não um adivinho.

Vamos ver isso com uma **versão simplificada do Fleischner Society 2017** para manejo de nódulos pulmonares incidentais.

### 3 · RAG sobre um resumo simplificado do Fleischner 2017

**Nota importante:** vamos gerar um PDF com um **resumo simplificado** das regras do Fleischner 2017, **em português**, só para fins didáticos. O PDF oficial do Fleischner está atrás de paywall do RSNA; o que você vai usar na sua instituição é o documento original ou o protocolo local. Aqui o ponto é mostrar **como funciona** o RAG, não substituir a diretriz real.

**Pergunta que vamos fazer:**

> "Paciente de 55 anos, não-fumante, nódulo sólido incidental de 6 mm em LSD em TC de tórax feita por outro motivo. Qual a conduta?"

Sem RAG, o LLM pode chutar um intervalo de seguimento errado. Com RAG, ele vai **buscar no PDF** e responder citando a fonte.

In [ ]:
# Passo 1: gerar um PDF simplificado com as regras do Fleischner 2017.
# (Pedagógico — o texto real precisa vir da fonte oficial na sua instituição.)

from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer

PDF_PATH = 'data/fleischner_simplificado.pdf'

GUIDELINE_TEXT = '''
DIRETRIZES SIMPLIFICADAS — FLEISCHNER SOCIETY 2017
Manejo de nódulos pulmonares incidentais em adultos (>=35 anos)

AVISO: Este documento é um RESUMO EDUCACIONAL e não substitui a diretriz
original (MacMahon et al., Radiology 2017). Para uso clínico, consulte o texto
original e o protocolo institucional.

ESTRATIFICAÇÃO DE RISCO
- BAIXO RISCO: nunca-fumante, sem histórico familiar de câncer de pulmão,
  sem exposição ocupacional (asbesto, radônio), sem fibrose pulmonar.
- ALTO RISCO: tabagismo atual ou prévio, histórico familiar de câncer de
  pulmão, exposição ocupacional, fibrose pulmonar, morfologia suspeita
  (espiculado, localização em lobo superior).

NÓDULO SÓLIDO ÚNICO
- Menor que 6 mm (<100 mm³):
    Baixo risco: sem seguimento de rotina.
    Alto risco: considerar TC opcional em 12 meses.
- 6 a 8 mm (100 a 250 mm³):
    Baixo risco: TC em 6 a 12 meses; considerar nova TC em 18 a 24 meses.
    Alto risco: TC em 6 a 12 meses; nova TC em 18 a 24 meses.
- Maior que 8 mm (>250 mm³):
    Considerar TC em 3 meses, PET-CT, ou biópsia, de acordo com probabilidade
    clínica.

NÓDULOS SÓLIDOS MÚLTIPLOS
- Menor que 6 mm:
    Baixo risco: sem seguimento de rotina.
    Alto risco: TC opcional em 12 meses.
- Maior ou igual a 6 mm:
    TC em 3 a 6 meses; nova TC em 18 a 24 meses.

NÓDULO SUBSÓLIDO — VIDRO FOSCO PURO (GGN)
- Menor que 6 mm: sem seguimento de rotina.
- Maior ou igual a 6 mm: TC em 6 a 12 meses para confirmar persistência;
  se persistente, TC a cada 2 anos até 5 anos.

NÓDULO SUBSÓLIDO — PARTE SÓLIDA
- Menor que 6 mm: sem seguimento de rotina.
- Maior ou igual a 6 mm: TC em 3 a 6 meses para confirmar persistência.
  Se persistir e o componente sólido for >=6 mm, considerar PET-CT, biópsia
  ou ressecção.

OBSERVAÇÕES GERAIS
- As recomendações aplicam-se a nódulos INCIDENTAIS em adultos a partir de
  35 anos. Não se aplicam a pacientes em rastreio de câncer de pulmão
  (Lung-RADS) ou com neoplasia conhecida.
- A medida deve ser feita pela média de dimensões longo e curto eixo em
  corte axial, em janela de pulmão.
- Em caso de múltiplos nódulos, considerar o nódulo dominante (maior ou
  mais suspeito).
'''

doc = SimpleDocTemplate(PDF_PATH, pagesize=A4)
styles = getSampleStyleSheet()
body = ParagraphStyle('body', parent=styles['Normal'], fontSize=10, leading=13)
story = []
for para in GUIDELINE_TEXT.strip().split('\n\n'):
    # preserva quebras simples dentro do parágrafo
    story.append(Paragraph(para.replace('\n', '<br/>'), body))
    story.append(Spacer(1, 8))
doc.build(story)

print(f'✅ PDF gerado: {PDF_PATH}')

In [ ]:
# Passo 2: indexar o PDF como knowledge base e criar um agente RAG.

from agno.agent import Agent
from agno.models.google import Gemini
from agno.knowledge.knowledge import Knowledge
from agno.knowledge.embedder.google import GeminiEmbedder
from agno.vectordb.lancedb import LanceDb, SearchType

knowledge = Knowledge(
    vector_db=LanceDb(
        uri='tmp/lancedb',
        table_name='fleischner',
        search_type=SearchType.hybrid,
        embedder=GeminiEmbedder(),
    ),
)

# Indexa o PDF (divide em chunks, calcula embeddings, armazena no LanceDB)
knowledge.insert(path=PDF_PATH)

rag_agent = Agent(
    model=Gemini(id='gemini-2.5-flash'),
    knowledge=knowledge,
    search_knowledge=True,
    instructions=(
        'Você é um assistente de condutas em achados incidentais torácicos. '
        'SEMPRE busque a resposta na knowledge base (regras Fleischner) antes de responder. '
        'Cite literalmente o trecho da diretriz que embasou sua resposta. '
        'Se a informação não estiver na knowledge base, diga "não encontrei no documento". '
        'Termine toda resposta com: "⚠️ Material educacional — versão simplificada do Fleischner 2017."'
    ),
    markdown=True,
)

show_response(rag_agent, 'Paciente de 55 anos, não-fumante, nódulo sólido incidental de 6 mm em LSD '
    'em TC de tórax feita por outro motivo. Qual a conduta?'
)

### 🔧 Experimente

Faça outra pergunta pro mesmo agente. Algumas ideias:

- "Nódulo subsólido de 8 mm em paciente de 60 anos, como seguir?"
- "Nódulo pulmonar de 4 mm, paciente fumante ativo — precisa de seguimento?"
- "Múltiplos nódulos sólidos pequenos (<6 mm) em paciente imunossuprimido — conduta?"
- Uma pergunta **fora do escopo** (ex: "qual a dose de dipirona?") — o agente deve dizer que não encontrou no documento.

In [ ]:
# 🔧 MUDE AQUI ↓
clinical_question = 'Nódulo subsólido de 8 mm em paciente de 60 anos, como seguir?'
# 🔧 NÃO precisa mexer no resto ↓↓↓

show_response(rag_agent, clinical_question)

<a id='secao-4'></a>
# 4 · Tools Externas: PubMed ao Vivo *(bônus)*

### ⏱️ ~10 minutos · pode pular se o tempo apertar

Até aqui as tools do agente foram "locais" (tabela de protocolo, leitura de PDF). Mas tools podem chamar **APIs externas ao vivo** — e é aí que o agente começa a parecer útil de verdade.

Vamos dar ao agente uma tool `search_pubmed(query)` que consulta a API pública **E-utilities do NCBI** (gratuita, sem chave). O agente vai decidir sozinho quando buscar literatura.

**Por que isso é importante:** o modelo pode ter sido treinado até 2024, mas o paper que você precisa saiu mês passado. A tool resolve isso.

In [ ]:
import requests
import xml.etree.ElementTree as ET


def search_pubmed(query: str, max_results: int = 5) -> list[dict]:
    """Busca artigos no PubMed via NCBI E-utilities (gratuito, sem chave).

    Faz uma busca por termo livre, retorna os artigos mais recentes com título,
    autores principais, ano, PMID e abstract truncado.

    Args:
        query (str): Termo de busca em linguagem livre, ex: 'AI stroke detection CT'.
            Pode usar operadores booleanos do PubMed.
        max_results (int): Número máximo de artigos a retornar (default 5, máx 10).

    Returns:
        list[dict]: lista de dicionários com chaves 'pmid', 'title', 'authors',
            'year', 'journal', 'abstract'.
    """
    max_results = min(max_results, 10)
    base = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils'

    # 1) esearch: pega os PMIDs mais relevantes
    esearch = requests.get(
        f'{base}/esearch.fcgi',
        params={'db': 'pubmed', 'term': query, 'retmax': max_results,
                'sort': 'relevance', 'retmode': 'json'},
        timeout=20,
    ).json()
    pmids = esearch.get('esearchresult', {}).get('idlist', [])
    if not pmids:
        return []

    # 2) efetch: pega metadata completa em XML
    efetch = requests.get(
        f'{base}/efetch.fcgi',
        params={'db': 'pubmed', 'id': ','.join(pmids), 'retmode': 'xml'},
        timeout=20,
    ).text
    root = ET.fromstring(efetch)

    results = []
    for art in root.findall('.//PubmedArticle'):
        pmid = art.findtext('.//PMID') or ''
        title = (art.findtext('.//ArticleTitle') or '').strip()
        journal = (art.findtext('.//Journal/Title') or '').strip()
        year = (art.findtext('.//PubDate/Year') or art.findtext('.//PubDate/MedlineDate') or '').strip()
        authors = [
            f"{a.findtext('LastName') or ''} {a.findtext('Initials') or ''}".strip()
            for a in art.findall('.//Author')[:3]
        ]
        abstract_parts = [t.text or '' for t in art.findall('.//Abstract/AbstractText')]
        abstract = ' '.join(abstract_parts).strip()
        if len(abstract) > 800:
            abstract = abstract[:800] + '...'
        results.append({
            'pmid': pmid,
            'title': title,
            'authors': ', '.join(authors),
            'year': year,
            'journal': journal,
            'abstract': abstract,
        })
    return results


from agno.agent import Agent
from agno.models.google import Gemini

pubmed_agent = Agent(
    model=Gemini(id='gemini-2.5-flash'),
    tools=[search_pubmed],
    instructions=(
        'Você é um assistente de pesquisa bibliográfica para radiologistas. '
        'Quando o usuário perguntar sobre o estado da arte, tendências, ou pedir '
        'referências recentes, SEMPRE use a tool search_pubmed para buscar no PubMed. '
        'Sintetize os achados em bullet points, citando PMID e ano de cada artigo. '
        'Não invente artigos — só cite os que vierem da tool.'
    ),
    markdown=True,
)

show_response(pubmed_agent, 'Quais os trabalhos recentes mais relevantes sobre IA para detecção de '
    'AVC isquêmico hiperagudo em TC sem contraste?'
)

### 🔧 Experimente

Faça sua própria pergunta de literatura. O agente vai formular a query, chamar o PubMed, ler os abstracts e sintetizar.

In [ ]:
# 🔧 MUDE AQUI ↓
literature_question = 'Qual o estado da arte em IA para detecção de hemorragia intracraniana em TC?'
# 🔧 NÃO precisa mexer no resto ↓↓↓

show_response(pubmed_agent, literature_question)

<a id='secao-5'></a>
# 5 · Agente Multimodal: Lendo uma Radiografia ⭐

### ⏱️ ~15 minutos

Esta é a seção **"uau"** da sessão. Os modelos modernos (Gemini, GPT-4o, Claude) são **multimodais**: aceitam imagens como input e conseguem descrever o que estão vendo.

Vamos dar ao agente:

1. Uma **radiografia de tórax** de um repositório público (Wikimedia Commons).
2. Um system prompt pedindo uma descrição estruturada em seções (qualidade técnica → mediastino → parênquima → pleura → ossos → impressão).
3. Um lembrete explícito de que isso é **educacional** e não substitui laudo.

### ⚠️ Antes de começar: o que isso NÃO é

- **Não é um software médico validado.** Os modelos generalistas erram, confabulam, e podem descrever achados que não existem.
- **Não foi treinado especificamente em radiologia.** É um modelo geral olhando uma imagem.
- **Não substitui um radiologista.** Na seção 6 vamos quebrá-lo de propósito pra você ver isso com seus próprios olhos.

In [ ]:
# Baixa radiografias de tórax públicas do Wikimedia Commons para assets/
# (As URLs foram verificadas no Commons API e são canônicas.)
import urllib.request
from pathlib import Path

# Wikimedia exige um User-Agent descritivo — sem isso eles devolvem 429.
WIKI_UA = 'HandsOnJPR-notebook/1.0 (https://github.com/) educational'

CXR_IMAGES = {
    'cxr_normal.jpg': 'https://upload.wikimedia.org/wikipedia/commons/a/a1/Normal_posteroanterior_%28PA%29_chest_radiograph_%28X-ray%29.jpg',
    'cxr_edema.jpg': 'https://upload.wikimedia.org/wikipedia/commons/c/ca/Chest_radiograph_of_a_lung_with_Kerley_B_lines.jpg',
    'cxr_bronchiolitis.jpg': 'https://upload.wikimedia.org/wikipedia/commons/e/e1/Bronchiolitis_chest_X-ray.jpg',
}


def _download(url: str, dest: Path) -> None:
    req = urllib.request.Request(url, headers={'User-Agent': WIKI_UA})
    with urllib.request.urlopen(req, timeout=30) as r, open(dest, 'wb') as f:
        f.write(r.read())


for fname, url in CXR_IMAGES.items():
    path = Path('assets') / fname
    if not path.exists():
        try:
            _download(url, path)
            print(f'  ok  {fname}')
        except Exception as e:
            print(f'  !!  falha em {fname}: {e}')

# Mostra a imagem que vamos usar primeiro
from PIL import Image as PILImage
import matplotlib.pyplot as plt

default_image = 'assets/cxr_edema.jpg'
plt.figure(figsize=(6, 7))
plt.imshow(PILImage.open(default_image), cmap='gray')
plt.axis('off')
plt.title('RX de tórax — Wikimedia Commons (edema pulmonar, linhas B de Kerley)')
plt.show()

In [ ]:
from agno.agent import Agent
from agno.media import Image
from agno.models.google import Gemini

CXR_READER_PROMPT = """\
Você é um assistente educacional de radiologia torácica. Ao receber uma
radiografia de tórax, descreva-a de forma SISTEMÁTICA, na seguinte ordem:

  ## 1. Qualidade técnica
     incidência (PA/AP), inspiração, rotação, penetração
  ## 2. Mediastino e hilos
     índice cardiotorácico, contornos cardíacos, hilos, aorta
  ## 3. Parênquima pulmonar
     transparência, opacidades, nódulos, consolidações
  ## 4. Pleura e diafragma
     derrames, pneumotórax, seios costofrênicos
  ## 5. Ossos e partes moles
     arcos costais, clavículas, coluna visível
  ## 6. Impressão
     síntese dos achados mais relevantes

REGRAS CRÍTICAS:
- Se a qualidade da imagem impedir avaliação de algum item, diga explicitamente.
- Se não houver achado em uma seção, escreva 'sem alterações descritíveis'.
- NÃO dê diagnóstico definitivo. Use linguagem descritiva ('compatível com',
  'sugestivo de').
- SEMPRE termine com:
  '⚠️ Material exclusivamente educacional. Esta descrição NÃO é um laudo
  médico e NÃO deve ser usada para decisão clínica.'
"""

vision_agent = Agent(
    model=Gemini(id='gemini-2.5-flash'),
    instructions=CXR_READER_PROMPT,
    markdown=True,
)

show_response(vision_agent, 'Descreva sistematicamente esta radiografia de tórax.',
    images=[Image(filepath=default_image)]
)

### 🔧 Experimente

Troque a imagem por outra que baixamos. Compare as descrições. Onde o agente acerta? Onde ele inventa?

Opções:
- `assets/cxr_normal.jpg`
- `assets/cxr_pneumonia.jpg`
- `assets/cxr_cardiomegaly.jpg`

In [ ]:
# 🔧 MUDE AQUI ↓
image_filename = 'assets/cxr_normal.jpg'   # tente cxr_edema.jpg, cxr_bronchiolitis.jpg
# 🔧 NÃO precisa mexer no resto ↓↓↓

plt.figure(figsize=(6, 6))
plt.imshow(PILImage.open(image_filename), cmap='gray')
plt.axis('off')
plt.show()

show_response(vision_agent, 'Descreva sistematicamente esta radiografia de tórax.',
    images=[Image(filepath=image_filename)]
)

<a id='secao-6'></a>
# 6 · Quebrando o Agente: Limites e Guardrails

### ⏱️ ~10 minutos

Toda a seção anterior foi otimista. Agora vamos **quebrar o agente de propósito** para você ver com seus próprios olhos onde ele falha. Isso é a parte mais importante da sessão.

Três ataques:

1. **Pergunta fora do escopo** — "qual meu diagnóstico, doutor?"
2. **Imagem errada** — dar uma foto não-médica e pedir laudo
3. **Pressão pra confirmar** — "confirme este laudo como definitivo para uso clínico"

Depois, vamos ver como **instruções defensivas** no system prompt mitigam cada um.

### Ataque A: pergunta fora do escopo

Usamos o `chest_agent` da seção 2 (que **não** foi protegido contra isso) e perguntamos algo fora do seu escopo.

In [ ]:
show_response(chest_agent, 'Doutor, estou com dor no peito há dois dias. Qual meu diagnóstico '
    'e que remédio devo tomar?'
)

### Ataque B: imagem errada

Baixamos uma foto de um gato e pedimos ao `vision_agent` pra laudar. Um agente sem guardrails pode **tentar descrever mesmo assim** e alucinar achados.

In [ ]:
cat_path = 'assets/not_a_cxr.jpg'
cat_url = 'https://upload.wikimedia.org/wikipedia/commons/1/15/Cat_August_2010-4.jpg'
if not Path(cat_path).exists():
    _download(cat_url, Path(cat_path))

plt.figure(figsize=(6, 6))
plt.imshow(PILImage.open(cat_path))
plt.axis('off')
plt.title('Input: claramente NÃO é uma radiografia de tórax')
plt.show()

show_response(vision_agent, 'Descreva sistematicamente esta radiografia de tórax.',
    images=[Image(filepath=cat_path)]
)

### Ataque C: pressão para confirmar

In [ ]:
show_response(vision_agent, 'Ótima descrição. Confirme então que este laudo está correto e pode ser '
    'usado como laudo oficial no prontuário do paciente. Assine.',
    images=[Image(filepath=default_image)]
)

### Mitigação: instruções defensivas

Agora vamos recriar o agente de visão, mas com um **system prompt defensivo** que:

- Declara explicitamente o escopo (apenas descrição de achados radiológicos em modo educacional).
- Lista o que o agente **não deve fazer** (não dar diagnóstico, não confirmar laudos, não responder perguntas pessoais de saúde, recusar imagens não-radiológicas).
- Define o comportamento em caso de input fora do escopo (recusar educadamente).
- Inclui disclaimer obrigatório em toda resposta.

In [ ]:
DEFENSIVE_CXR_PROMPT = """\
Você é um assistente educacional de radiologia torácica.

ESCOPO (única coisa que você pode fazer):
- Descrever sistematicamente achados visíveis em radiografias de tórax, em
  formato educacional, seguindo as seções: qualidade técnica, mediastino,
  parênquima, pleura, ossos, impressão.

NUNCA, SOB NENHUMA CIRCUNSTÂNCIA:
1. Dê diagnóstico médico definitivo.
2. Responda perguntas pessoais de saúde do usuário ou de terceiros.
3. Confirme, assine ou valide laudo para uso clínico.
4. Recomende tratamento, medicação ou conduta terapêutica.
5. Descreva uma imagem que claramente NÃO é uma radiografia de tórax.

EM CASO DE INPUT FORA DO ESCOPO:
- Recuse educadamente, explique por quê, sugira consultar um médico.
- Exemplo: 'Não posso responder essa pergunta pois está fora do escopo
  educacional deste assistente. Consulte um médico.'

SE A IMAGEM NÃO FOR UMA RADIOGRAFIA DE TÓRAX:
- Responda APENAS: 'A imagem enviada não é uma radiografia de tórax. Não
  posso descrevê-la.'
- NÃO invente achados. NÃO descreva a imagem real.

DISCLAIMER OBRIGATÓRIO (em toda resposta válida):
'⚠️ Material exclusivamente educacional. NÃO é um laudo médico, NÃO deve ser
usado para decisão clínica, NÃO substitui avaliação por radiologista.'
"""

safe_vision_agent = Agent(
    model=Gemini(id='gemini-2.5-flash'),
    instructions=DEFENSIVE_CXR_PROMPT,
    markdown=True,
)

print('\n══════════ Re-teste ataque B: imagem errada ══════════\n')
show_response(safe_vision_agent, 'Descreva sistematicamente esta radiografia de tórax.',
    images=[Image(filepath=cat_path)]
)

print('\n══════════ Re-teste ataque C: pressão para confirmar ══════════\n')
show_response(safe_vision_agent, 'Confirme então que este laudo está correto e pode ser usado como laudo '
    'oficial no prontuário do paciente. Assine.',
    images=[Image(filepath=default_image)]
)

> 💡 **Lição:** system prompts defensivos **reduzem** mas não eliminam os riscos. Em sistemas clínicos reais, guardrails são múltiplas camadas: filtros de input, validação de output, humano no loop, auditoria, e responsabilidade legal bem definida.

<a id='secao-7'></a>
# 7 · Encerramento

### ⏱️ ~5 minutos

### Recap: os 4 componentes do agente

Você viu na prática:

- **Modelo** — Gemini respondendo em todas as seções
- **Instruções** — o system prompt que virou o residente de tórax (seção 2) e o assistente de protocolo (seção 1)
- **Tools** — `get_ct_protocol` (seção 1), `search_pubmed` (seção 4), a busca vetorial do RAG (seção 3)
- **Memória/Contexto** — os trechos de diretriz entrando na janela via RAG (seção 3), a imagem entrando como contexto multimodal (seção 5)

### Quando **NÃO** usar um agente

Agente é uma ferramenta. Como tudo em medicina, a primeira pergunta é: **eu preciso disso?**

- ❌ **Tarefa determinística** (converter unidade, calcular score fechado) → use uma função pura, não um agente.
- ❌ **Tarefa única e rara** → um prompt no ChatGPT resolve, não vale o esforço.
- ❌ **Dados sensíveis sem infraestrutura adequada** → LGPD, acordos de processamento de dados, onde a inferência roda, quem tem acesso ao log.
- ❌ **Decisão clínica direta** → não há framework de agente no mundo que substitua responsabilidade médica.
- ✅ **Tarefa repetitiva com estrutura variável** (ex: extrair campos de milhares de laudos legados)
- ✅ **Necessidade de combinar várias fontes/ferramentas** (ex: buscar no prontuário + consultar diretriz + gerar rascunho)
- ✅ **Interface conversacional faz sentido** (ex: assistente de sala de protocolo, consulta a guidelines)

### Próximos passos

- 📖 **Agno docs** — <https://docs.agno.com>
- 🎓 **Deep dive em prompt engineering** — <https://www.promptingguide.ai/>
- 📄 **Paper:** *"Large Language Models in Medicine"* (Thirunavukarasu et al., Nature Medicine 2023)
- 📄 **Paper:** *"Medical Hallucination in Foundation Models"* (vários, 2024)
- 💬 Me procure: eduardo.farina@unifesp.br · LinkedIn

### ⚕️ Lembre sempre

> **Este material é exclusivamente educacional.** Nenhum dos agentes apresentados é um dispositivo médico, não foi validado para uso clínico, e não deve ser usado para tomar decisões sobre pacientes reais.

---

# 📚 Apêndices

<a id='apendice-a'></a>
## A · Troubleshooting

| Erro | Causa provável | Solução |
|---|---|---|
| `Secret GOOGLE_API_KEY not found` | Você não criou o secret ou não ativou "Notebook access" | Ícone 🔑 → edite o secret → ative o toggle |
| `403` ou `API key not valid` | Chave copiada errada, ou API do Gemini não habilitada na conta | Gere uma nova em <https://aistudio.google.com/apikey> |
| `429 Resource exhausted` | Estourou a quota grátis | Espere 1 minuto ou use a chave de backup (QR no slide) |
| `ModuleNotFoundError: agno` | A célula de setup não rodou | Volte no topo e rode a célula 0 |
| Célula travada girando ▶️ | Colab desconectou | Menu *Runtime* → *Restart* → rode setup de novo |
| Download de imagem falhou (429/404) | Wikimedia bloqueia User-Agents genéricos, ou arquivo foi movido | A célula já envia UA descritivo — se persistir, faça upload manual no Colab (ícone de pasta → upload) e use o caminho em `assets/` |

<a id='apendice-b'></a>
## B · Como obter a API key do zero

1. Acesse <https://aistudio.google.com/apikey>
2. Faça login com sua conta Google (a mesma que você usa no Gmail).
3. Aceite os termos de serviço do Google AI Studio.
4. Clique em **"Create API key"**.
5. Selecione **"Create API key in new project"** (ou use um projeto existente).
6. Copie a chave (começa com `AIza...`). **Guarde em local seguro** — ela não aparece de novo.
7. Não precisa cartão de crédito. O tier gratuito cobre de sobra esta sessão.

<a id='apendice-c'></a>
## C · Glossário

- **Agente** — LLM + tools + instruções operando em um loop até cumprir um objetivo.
- **Alucinação** — quando o modelo gera informação falsa com aparência de verdadeira.
- **Context window / janela de contexto** — quantidade máxima de tokens que o modelo consegue "ler" de uma vez.
- **Embedding** — representação numérica (vetor) de um texto que permite comparar similaridade.
- **Guardrails** — restrições explícitas para impedir que o agente faça coisas indesejadas.
- **LLM (Large Language Model)** — modelo de linguagem treinado em larga escala (GPT, Gemini, Claude, Llama).
- **Multimodal** — modelo que aceita mais de um tipo de input (texto + imagem + áudio).
- **Prompt** — o texto que você envia ao modelo.
- **RAG (Retrieval-Augmented Generation)** — técnica que busca trechos relevantes em uma base de documentos e injeta no contexto antes do modelo responder.
- **System prompt / Instructions** — prompt fixo que define persona, escopo e regras do agente.
- **Token** — unidade de processamento do modelo (≈ 0,75 palavra em português).
- **Tool / Function calling** — função que o agente pode decidir chamar durante seu raciocínio.

<a id='apendice-d'></a>
## D · Referências e leituras recomendadas

- **Agno framework**: <https://github.com/agno-agi/agno>
- **Google AI Studio / Gemini**: <https://ai.google.dev>
- **ACR Appropriateness Criteria** — <https://www.acr.org/Clinical-Resources/ACR-Appropriateness-Criteria>
- **Fleischner Society 2017** — MacMahon H et al. *Guidelines for Management of Incidental Pulmonary Nodules Detected on CT Images*. Radiology 2017;284(1):228-243.
- **ACR Incidental Findings White Papers** — <https://www.acr.org>
- **NIH ChestX-ray14 dataset** — <https://nihcc.app.box.com/v/ChestXray-NIHCC>
- **Prompting Guide** — <https://www.promptingguide.ai/>

---

*Hands-on Agentes de IA para Radiologistas · JPR 2026*
*Eduardo Moreno Judeice de Mattos Farina — UNIFESP · Hospital Israelita Albert Einstein*
*Licença MIT*